# LL-DSV-UNet — Interactive Demo

**Low-Light Image Enhancement with Multi-Level Fusion Deep Supervision**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/LL-DSV-UNet/blob/main/notebooks/demo.ipynb)

This notebook lets you:
1. Install dependencies and clone the repo
2. Load the pretrained LL-DSV-UNet model
3. Enhance your own low-light images
4. Evaluate on the LOL Eval15 benchmark
5. Fine-tune on custom data

---

## 1. Setup

In [ ]:
!git clone https://github.com/yourusername/LL-DSV-UNet.git
%cd LL-DSV-UNet
!pip install -r requirements.txt -q

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from lldsvunet import build_lldsvunet, custom_loss, evaluate_model
from lldsvunet.utils.visualization import plot_sample_pairs

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Build / Load Model

In [ ]:
# Build architecture
model = build_lldsvunet(input_shape=(200, 300, 3))
model.summary()
print(f'Parameters: {model.count_params():,}')

In [ ]:
# Load pretrained weights (uncomment after downloading .h5)
# model = tf.keras.models.load_model(
#     'LL-DSV-UNet-final.h5',
#     custom_objects={'custom_loss': custom_loss}
# )

## 3. Enhance Your Own Image

In [ ]:
from google.colab import files
from lldsvunet.data.dataset import load_single_image

print('Upload a low-light image:')
uploaded = files.upload()

for filename in uploaded:
    img = load_single_image(filename, target_size=(200, 300))
    enhanced = model.predict(img, verbose=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(np.clip(img[0], 0, 1))
    axes[0].set_title('Low-light Input', fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(np.clip(enhanced[0], 0, 1))
    axes[1].set_title('Enhanced (LL-DSV-UNet)', fontsize=13)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 4. Evaluate on LOL Eval15

In [ ]:
# Uncomment after placing LOL eval15 data:
# from lldsvunet.data.dataset import load_images_from_directory
# x_eval = load_images_from_directory(['data/lol_dataset/eval15/low'])
# y_eval = load_images_from_directory(['data/lol_dataset/eval15/high'])
# y_pred = model.predict(x_eval)
# scores = evaluate_model(y_eval, y_pred)
# print(f"PSNR : {scores['psnr']:.2f} dB")
# print(f"SSIM : {scores['ssim']:.4f}")
# print(f"LPIPS: {scores['lpips']:.4f}")

## 5. Fine-tune on Your Own Dataset

```python
model = tf.keras.models.load_model(
    'LL-DSV-UNet-final.h5',
    custom_objects={'custom_loss': custom_loss}
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6, beta_2=0.99),
    loss=custom_loss
)
model.fit(x_custom, y_custom, batch_size=16, epochs=200, validation_split=0.1)
```